# Linear head timing

Reproduces the Head fit column of Table I. The models give features of different dimension, from 384 for SwinBrain to 9,664 for the un-CNN, so the linear head is timed separately from the encoders.

The features are the NKI features of the main benchmark, saved by `01_main_benchmark.ipynb` with `DATASET = "NKI"`. On one CPU thread, for each model and task, the full head fitting of the main benchmark, the inner fivefold selection over the 30 values of $\alpha$ or $C$ followed by the refit, is timed on the training set of the first outer fold, about 860 subjects, over 3 repetitions. Also timed: one fit at the selected value, and prediction one subject at a time on the test fold. The paper run used one SLURM job with 2 CPUs, submitted by `slurm/head_timing.sh`.

In [ ]:
REPS = 3
FOLDS = [0]
MODELS = ["Untrained_CNN", "AnatCL_Global", "AnatCL_Local", "BrainIAC", "SwinBrain", "3D-Neuro-SimCLR",
          "FS_Schaefer", "FS_aparc"]

In [ ]:
import os

for var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[var] = "1"

import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score, mean_absolute_error

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

import heads
from config import OUTPUT_DIR
from machine import system_info

FEATURES = OUTPUT_DIR / "features"
OUT = OUTPUT_DIR / "head_timing"
OUT.mkdir(parents=True, exist_ok=True)
info = system_info()
pd.DataFrame([info]).to_csv(OUT / "system_info.csv", index=False)
pd.Series(info)

## Features and folds

The outer folds are those of the main benchmark.

In [ ]:
labels = np.load(FEATURES / "NKI_labels.npz")
tasks = {"sex": (slice(None), labels["sex"]), "age": (slice(None), labels["age"])}
if len(labels["bmi_idx"]):
    tasks["bmi"] = (labels["bmi_idx"], labels["bmi"])
folds = {task: heads.make_folds(y, task) for task, (_, y) in tasks.items()}
print({task: len(y) for task, (_, y) in tasks.items()})

## Timing

`select_fit` is `heads.select_and_fit`, the 150 inner fits and the refit of the benchmark; `single_fit` is one fit at the selected value; `predict` is seconds per image. A fit on 50 subjects warms up each model and task untimed. The outer test fold score is kept to check that the timed head is the benchmark head.

In [ ]:
rows = []
for name in MODELS:
    X_all = np.load(FEATURES / f"NKI_{name}.npy")
    for task, (idx, y) in tasks.items():
        X = X_all[idx]
        tr0 = folds[task][0][0]
        heads.fit_head(X[tr0[:50]], y[tr0[:50]], task, 1.0)
        for k in FOLDS:
            tr, te = folds[task][k]
            base = {"model": name, "task": task, "fold": k, "n_train": len(tr), "n_features": X.shape[1]}
            for rep in range(REPS):
                t0 = time.perf_counter()
                model, lam, nc = heads.select_and_fit(X[tr], y[tr], task)
                rows.append({**base, "stage": "select_fit", "rep": rep, "seconds": time.perf_counter() - t0,
                             "lam": lam, "nonconv": nc})
            for rep in range(REPS):
                t0 = time.perf_counter()
                _, converged = heads.fit_head(X[tr], y[tr], task, lam)
                rows.append({**base, "stage": "single_fit", "rep": rep, "seconds": time.perf_counter() - t0,
                             "lam": lam, "nonconv": int(not converged)})
            for rep in range(REPS):
                t0 = time.perf_counter()
                for i in te:
                    heads.predict_head(model, X[i:i + 1])
                rows.append({**base, "stage": "predict", "rep": rep, "seconds": (time.perf_counter() - t0) / len(te),
                             "lam": lam, "nonconv": 0})
            out = heads.predict_head(model, X[te])
            score = (balanced_accuracy_score(y[te], (out >= 0).astype(int)) if task == "sex"
                     else mean_absolute_error(y[te], out))
            rows.append({**base, "stage": "test_fold_score", "rep": 0, "seconds": np.nan, "lam": lam, "nonconv": 0,
                         "score": score})
            fit_s = np.mean([r["seconds"] for r in rows[-(3 * REPS + 1):] if r["stage"] == "select_fit"])
            print(f"{name:<16} {task:<4} fold {k}: d={X.shape[1]:>5}  select_fit {fit_s:7.2f} s  lam {lam:.3g}  score {score:.3f}")

raw = pd.DataFrame(rows)
raw.to_csv(OUT / "raw.csv", index=False)

## Table I

Head fit is the mean `select_fit` time over sex, age and BMI. It is a one off cost per task and is not part of the Total of Table I.

In [ ]:
timed = raw[raw.stage != "test_fold_score"]
summary = (timed.groupby(["model", "task", "stage"], sort=False)
           .agg(n_features=("n_features", "first"), n_train=("n_train", "mean"),
                seconds_mean=("seconds", "mean"), seconds_std=("seconds", "std"))
           .reset_index())
summary.to_csv(OUT / "summary.csv", index=False)

fit = summary[summary.stage == "select_fit"].pivot(index="model", columns="task", values="seconds_mean").loc[MODELS]
fit["Head fit (s)"] = fit[list(tasks)].mean(axis=1)
fit.insert(0, "Dim.", summary.groupby("model")["n_features"].first())
fit.round(1)

In [ ]:
single = summary[summary.stage == "single_fit"]["seconds_mean"]
predict = summary[summary.stage == "predict"]["seconds_mean"]
print(f"single fit {single.min():.3f} to {single.max():.2f} s, prediction {1e3 * predict.mean():.2f} ms per image")

## Check against the main benchmark

The test fold scores must equal those of the benchmark on the same folds.

In [ ]:
bench = pd.read_csv(OUTPUT_DIR / "main_benchmark" / "NKI_folds.csv")
bench = bench[bench.metric.isin(["balanced_accuracy", "mae"])][["model", "task", "fold", "value"]]
check = raw[raw.stage == "test_fold_score"].merge(bench, on=["model", "task", "fold"])
print(f"{len(check)} model and task pairs, max abs difference {(check.score - check.value).abs().max():.2g}")